# Qwen2-VL QLoRA fine-tune on Kaggle GPU

**Before running:** Notebook menu → Settings → **Accelerator = GPU T4 x2** (or P100), and **Internet = On**.

This notebook clones your code from GitHub and runs `train.py` on the GPU. Checkpoints land in `/kaggle/working` and survive to the output tab.

In [ ]:
# 1) Sanity-check the GPU
!nvidia-smi

In [ ]:
# 2) Clone your repo fresh every run (remove any persisted copy so we always get latest code).
REPO = 'https://github.com/smshaqib/qwen2vl-finetune.git'
import shutil, os
shutil.rmtree('/kaggle/working/repo', ignore_errors=True)
!git clone --depth 1 $REPO /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

In [ ]:
# 3) Deps. Kaggle's API always hands out a P100 (sm_60); the image's torch 2.10 dropped
#    Pascal -> "no kernel image". Fix below, three parts:
#    (a) install an official torch cu121 build that still ships sm_60 kernels;
#    (b) PIN transformers/peft to versions that support Qwen2-VL AND torch 2.4.1
#        (the newest transformers imports modules that need a newer torch);
#    (c) remove the too-old torchao that breaks PEFT's LoRA setup.
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
with open("/tmp/con.txt", "w") as f:
    f.write("torch==2.4.1\ntorchvision==0.19.1\n")
!pip install -q -c /tmp/con.txt transformers==4.49.0 accelerate==1.0.1 peft==0.13.2 datasets qwen-vl-utils
!pip uninstall -y -q torchao 2>/dev/null; echo "torchao removed (ok if absent)"

# Verify torch runs an actual CUDA kernel on this GPU, and transformers imports cleanly
import torch
print("torch", torch.__version__, "| GPU", torch.cuda.get_device_name(0),
      "| capability", torch.cuda.get_device_capability(0))
_x = torch.randn(8, 8, device="cuda"); print("CUDA smoke-test OK ->", round((_x @ _x).sum().item(), 3))
import transformers, peft
print("transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
# 4) (Optional) HF login if you later use a gated model/dataset.
# from kaggle_secrets import UserSecretsClient
# from huggingface_hub import login
# login(UserSecretsClient().get_secret('HF_TOKEN'))

In [ ]:
# 5) Smoke test: train on 8 samples (~1-2 min) to shake out errors cheaply.
#    Once this passes, switch to: --config config.yaml  for the real run.
!python train.py --config config.debug.yaml

In [ ]:
# 6) Inference sanity-check: classify a few TEST memes with the trained adapter.
import os, torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel
from qwen_vl_utils import process_vision_info

BASE = 'Qwen/Qwen2-VL-2B-Instruct'
ADAPTER = '/kaggle/working/qwen2vl-meme-lora-debug'   # use qwen2vl-meme-lora for the full run
TEST_DIR = '/kaggle/input/meme-data/MemeDecode/Test/Image'
SYSTEM = ("You are a meme classifier. Look at the meme image and reply with exactly one "
          "of these categories and nothing else: Neutral, Genders, Religion, Politics.")

proc = AutoProcessor.from_pretrained(ADAPTER)
model = Qwen2VLForConditionalGeneration.from_pretrained(BASE, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(model, ADAPTER).eval()

def classify(img_path):
    msgs = [{'role':'system','content':SYSTEM},
            {'role':'user','content':[{'type':'image','image':img_path},
                                      {'type':'text','text':'Classify this meme into exactly one category.'}]}]
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    imgs, vids = process_vision_info(msgs)
    inp = proc(text=[text], images=imgs, videos=vids, return_tensors='pt').to(model.device)
    out = model.generate(**inp, max_new_tokens=8, do_sample=False)
    return proc.batch_decode(out[:, inp.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

for name in sorted(os.listdir(TEST_DIR))[:5]:
    print(f"{name} -> {classify(os.path.join(TEST_DIR, name))}")